# 01 — Generate all-central satellite catalogs (TNG100 & TNG50, z=0 and z=0.05)

Fresh-start generator (own `notebooks2/` + `data2/`). For **TNG100 and TNG50** at **z = 0** and
**z = 0.05**, for **all central galaxies** (no host halo-mass window), it computes each satellite's
**projected azimuthal angle** $\alpha\in[0,90]$ from the host major axis and records the
**3D distance from the central** (physical kpc and in units of $R_{200c}$). **No radius cut is
applied** — every satellite above the mass floor is written, so downstream notebooks can toggle a
radius cut themselves.

**Selections (all configurable in the config cell):**
* hosts: **all centrals** with reliable g+i SDSS-Sersic morphology and central $M_* > 10^7\,M_\odot$
  (`CENTRAL_LOGMSTAR_MIN`); no halo-mass window (`HOST_LOGM200 = (None, None)`).
* satellites: $M_{*,\rm sat} > 10^7\,M_\odot$ (`SAT_LOGM_MIN`), so notebook 02 can sub-select
  $>10^8$ for TNG100 and keep $>10^7$ for TNG50 from the same files.

> **Redshift note.** "z = 0.5" in the request is interpreted as **z = 0.05** (snapshot 98) — the
> redshift with SDSS-SKIRT mocks (needed for the host major axis), matching the rest of the
> project. A genuine z = 0.5 would need a different snapshot and SKIRT mocks that are not set up
> here; change the `REDSHIFTS`/`snap` mapping if you truly want it.

**Runs only where the raw TNG catalogs are mounted (the Binder instance).** Requires
`illustris_python` and `h5py`. Masses are physical $M_\odot$; little-h $= 0.6774$ (see
`../docs/conventions.md`). Writes to `../data2/<sim>/<z>/`.

In [ ]:
import os
import numpy as np
import pandas as pd
import h5py
import illustris_python as il

## Configuration

Loop over both simulations and both redshifts. Edit the cuts here; the defaults implement
"all centrals above $10^7\,M_\odot$, satellites above $10^7\,M_\odot$, no radius cut".

In [ ]:
# simulations and redshifts to build (the notebook loops over the full product)
SIMS      = ["TNG100", "TNG50"]
REDSHIFTS = ["z0", "z0p05"]                 # z0 -> snap 99 (z=0); z0p05 -> snap 98 (z=0.05)

_SIM_DIR  = {"TNG100": "L75n1820TNG", "TNG50": "L35n2160TNG"}
_LBOX_MPC_H = {"TNG100": 75.0, "TNG50": 35.0}
_SNAP     = {"z0": 99, "z0p05": 98}
_ZVAL     = {"z0": 0.0, "z0p05": 0.0485}    # redshift -> physical distance scale factor

# raw data root on Binder: ~/SimulationData/<sim_dir>
TNG_PARENT = os.path.expanduser("~/SimulationData")

h = 0.6774                                  # IllustrisTNG little-h (Planck 2015)

# --- selections (physical M_sun) ---
CENTRAL_LOGMSTAR_MIN = 7.0                  # keep centrals (hosts) with log10 M*_central > this
SAT_LOGM_MIN         = 7.0                  # keep satellites with log10 M*_sat > this
HOST_LOGM200         = (None, None)         # optional (min, max) log10 M200c host window; None = all centrals
SN_MIN               = 2.5                  # SDSS-Sersic S/N-per-pixel reliability floor

# NO radius cut here: satellites at all radii are written (downstream toggles its own cut).
# The catalog stores d_3d_kpc and d_r200_3d so any aperture can be applied later.

DATA_ROOT = "../data2"
CAT_TAG   = f"logM{SAT_LOGM_MIN:.2f}"       # e.g. 'logM7.00'
print("build:", SIMS, "x", REDSHIFTS, "| central M* >", CENTRAL_LOGMSTAR_MIN,
      "| sat M* >", SAT_LOGM_MIN, "| host window", HOST_LOGM200)

## Helpers

Identical geometry/quenched machinery to the original `notebooks/01`: broadcast each host's
Sersic major-axis angle to its satellites, wrap separations with the minimum-image convention,
fold the azimuth into $[0,90]$, and flag quenched satellites via the Martín-Navarro+ 2021 SFMS.

In [ ]:
def host_morph_to_satellites(sdss, first_sub, n_subs, subfind_ids, n_sub):
    '''Look up each host's Sersic morphology and broadcast it to its satellites.'''
    theta = np.zeros(n_sub); flag = np.zeros(n_sub); sflag = np.zeros(n_sub); sn = np.zeros(n_sub)
    theta_all = np.asarray(sdss["sersic_theta"]) * 180.0 / np.pi
    flag_all  = np.asarray(sdss["flag"]); sflag_all = np.asarray(sdss["flag_sersic"])
    sn_all    = np.asarray(sdss["sn_per_pixel"])
    for k in range(len(first_sub)):
        c = first_sub[k]
        row = np.where(subfind_ids == c)[0]
        if row.size == 0:                                  # host has no SKIRT entry -> flag bad
            flag[c + 1:c + n_subs[k]] = 1
            continue
        r = row[0]; sl = slice(c + 1, c + n_subs[k])
        theta[sl] = theta_all[r]; flag[sl] = flag_all[r]; sflag[sl] = sflag_all[r]; sn[sl] = sn_all[r]
    return theta, flag, sflag, sn

def wrap_min_image(d, L):
    '''Minimum-image periodic wrap of a separation vector.'''
    return d - L * np.round(d / L)

def angle_from_major_axis(phi_sat_deg, pa_host_deg):
    '''Fold |phi_sat - PA_host| into [0, 90] using the disk 180-deg / mirror symmetry.'''
    phi = np.mod(phi_sat_deg, 360.0)
    d1 = np.abs(phi - np.mod(pa_host_deg, 360.0))
    d2 = np.abs(phi - np.mod(pa_host_deg + 180.0, 360.0))
    d1 = np.minimum(d1, 360.0 - d1); d2 = np.minimum(d2, 360.0 - d2)
    m = np.minimum(d1, d2)
    return np.minimum(m, 180.0 - m)

def quenched_flag(logmstar_phys, sfr):
    '''1 if >= 1 dex below the SDSS SFMS (Martin-Navarro+ 2021), else 0.'''
    sfr_ms = 10.0 ** (0.75 * logmstar_phys - 7.5)
    return (sfr < sfr_ms / 10.0).astype(int)

## Build one catalog per (simulation, redshift)

`process` loads the group/subhalo catalogs and SKIRT morphology for one snapshot, selects **all
centrals** (optionally within a halo-mass window) with reliable Sersic fits and central
$M_*>$ `CENTRAL_LOGMSTAR_MIN`, flags their satellites, computes $\alpha$ and the 3D/projected
host-centric distances, applies **only** the satellite mass floor (no radius cut), and writes the
catalog to `../data2/<sim>/<z>/tng_satellites_allcentrals_<tag>.csv`.

In [ ]:
def process(sim, redshift):
    sim_dir = _SIM_DIR[sim]; snap = _SNAP[redshift]; A = 1.0 / (1.0 + _ZVAL[redshift])
    Lbox = _LBOX_MPC_H[sim] * 1000.0 / h                         # physical kpc
    root = os.path.join(TNG_PARENT, sim_dir)
    basePath = os.path.join(root, "output")
    sp = f"snapnum_{snap:03d}"
    morph_g = os.path.join(root, f"postprocessing/skirt_images/sdss/{sp}/morphs_g.hdf5")
    morph_i = os.path.join(root, f"postprocessing/skirt_images/sdss/{sp}/morphs_i.hdf5")
    subfind_id_path = os.path.join(root, f"postprocessing/skirt_images/sdss/{sp}/subfind_ids.txt")

    gc = os.path.join(basePath, f"groups_{snap:03d}", f"fof_subhalo_tab_{snap:03d}.0.hdf5")
    if not (os.path.exists(gc) and os.path.exists(morph_g)):
        print(f"[skip] {sim} {redshift}: data not found at {root}")
        return None

    groups = il.groupcat.loadHalos(basePath, snap,
        fields=["GroupFirstSub", "Group_M_Crit200", "Group_R_Crit200", "GroupNsubs"])
    subhalos = il.groupcat.loadSubhalos(basePath, snap,
        fields=["SubhaloGrNr", "SubhaloMassType", "SubhaloCM", "SubhaloSFR"])
    n_sub = len(subhalos["SubhaloGrNr"])
    sdss_g = h5py.File(morph_g, "r"); sdss_i = h5py.File(morph_i, "r")
    subfind_ids = np.loadtxt(subfind_id_path)

    # per-subhalo stellar masses (both conventions)
    with np.errstate(divide="ignore"):
        mstar_phys = np.log10(subhalos["SubhaloMassType"][:, 4] * 1e10 / h)      # physical M_sun
        mstar_hinv = np.log10(subhalos["SubhaloMassType"][:, 4] * 1e10)          # h^-1 M_sun
    sat_sfr = np.asarray(subhalos["SubhaloSFR"])

    # ---- host (central) selection: ALL centrals, optional halo window + central M* floor ----
    with np.errstate(divide="ignore"):
        host_logm200_phys = np.log10(groups["Group_M_Crit200"] * 1e10 / h)
    first_all = groups["GroupFirstSub"]
    valid = first_all >= 0                                          # group has a central subhalo
    cen_mstar = np.full(len(first_all), -np.inf)
    cen_mstar[valid] = mstar_phys[first_all[valid]]
    host_sel = valid & (cen_mstar > CENTRAL_LOGMSTAR_MIN)
    lo, hi = HOST_LOGM200
    if lo is not None: host_sel &= host_logm200_phys > lo
    if hi is not None: host_sel &= host_logm200_phys < hi

    first_sub = groups["GroupFirstSub"][host_sel]
    n_subs    = groups["GroupNsubs"][host_sel]
    host_m200_phys_all = host_logm200_phys[host_sel]
    host_m200_hinv_all = np.log10(groups["Group_M_Crit200"][host_sel] * 1e10)
    cen_mstar_all      = cen_mstar[host_sel]

    # per-satellite arrays, filled only for satellites of selected centrals
    sat_mask       = np.zeros(n_sub, dtype=bool)
    host_id_arr    = np.full(n_sub, -1, dtype=int)
    host_center    = np.zeros((n_sub, 3))
    host_r200_arr  = np.zeros(n_sub)
    host_m200_phys = np.zeros(n_sub); host_m200_hinv = np.zeros(n_sub)
    host_mstar_arr = np.zeros(n_sub)
    for k in range(len(first_sub)):
        c = first_sub[k]; sl = slice(c + 1, c + n_subs[k])
        sat_mask[sl] = True
        host_id_arr[sl] = k
        host_center[sl] = subhalos["SubhaloCM"][c] / h
        host_r200_arr[sl] = groups["Group_R_Crit200"][host_sel][k] / h
        host_m200_phys[sl] = host_m200_phys_all[k]; host_m200_hinv[sl] = host_m200_hinv_all[k]
        host_mstar_arr[sl] = cen_mstar_all[k]

    # ---- host Sersic orientation + quality (mean of g and i) ----
    tg, fg, sfg, sng = host_morph_to_satellites(sdss_g, first_sub, n_subs, subfind_ids, n_sub)
    ti, fi, sfi, sni = host_morph_to_satellites(sdss_i, first_sub, n_subs, subfind_ids, n_sub)
    host_theta = 0.5 * (tg + ti)
    host_good  = ((fg == 0) & (sfg == 0) & (sng > SN_MIN) &
                  (fi == 0) & (sfi == 0) & (sni > SN_MIN))

    # ---- azimuthal angle alpha and 3D / projected host-centric distances ----
    rel = wrap_min_image(subhalos["SubhaloCM"] / h - host_center, Lbox)          # comoving kpc
    phi_2d = np.degrees(np.arctan2(rel[:, 1], rel[:, 0]))
    alpha = angle_from_major_axis(phi_2d, host_theta)
    _d3d = np.linalg.norm(rel, axis=1); _dpr = np.hypot(rel[:, 0], rel[:, 1])
    d_3d_kpc = _d3d * A; d_proj_kpc = _dpr * A
    with np.errstate(invalid="ignore", divide="ignore"):
        gr = host_r200_arr > 0
        d_r200_3d   = np.where(gr, _d3d / host_r200_arr, np.nan)
        d_r200_proj = np.where(gr, _dpr / host_r200_arr, np.nan)

    # ---- select satellites (mass floor only; NO radius cut) and write ----
    sel = sat_mask & host_good & (mstar_phys > SAT_LOGM_MIN)
    df = pd.DataFrame({
        "host_id":          host_id_arr[sel],
        "mstar_phys":       mstar_phys[sel],
        "mstar_hinv":       mstar_hinv[sel],
        "host_mstar_phys":  host_mstar_arr[sel],       # central (host) stellar mass, log10 M_sun
        "host_m200_phys":   host_m200_phys[sel],
        "host_m200_hinv":   host_m200_hinv[sel],
        "sfr":              sat_sfr[sel],
        "alpha":            alpha[sel],
        "d_3d_kpc":         d_3d_kpc[sel],
        "d_proj_kpc":       d_proj_kpc[sel],
        "d_r200_3d":        d_r200_3d[sel],
        "d_r200_proj":      d_r200_proj[sel],
        "quenched":         quenched_flag(mstar_phys[sel], sat_sfr[sel]),
    })
    assert df.alpha.between(0, 90).all(), "alpha outside [0, 90]!"

    outdir = os.path.join(DATA_ROOT, sim.lower(), redshift)
    os.makedirs(outdir, exist_ok=True)
    out = os.path.join(outdir, f"tng_satellites_allcentrals_{CAT_TAG}.csv")
    df.to_csv(out, index=False)
    print(f"[ok] {sim:6s} {redshift:5s}: centrals={host_sel.sum():6d}  "
          f"hosts_w_sat={df.host_id.nunique():5d}  sats={len(df):6d}  "
          f"f_q={df.quenched.mean():.3f}  -> {out}")
    sdss_g.close(); sdss_i.close()
    return dict(sim=sim, z=redshift, n_central=int(host_sel.sum()),
                n_host=int(df.host_id.nunique()), n_sat=len(df), path=out)

## Run the full product (both simulations, both redshifts)

In [ ]:
summary = []
for sim in SIMS:
    for z in REDSHIFTS:
        try:
            r = process(sim, z)
            if r: summary.append(r)
        except Exception as e:
            print(f"[error] {sim} {z}: {type(e).__name__}: {e}")

print("\n=== summary ===")
for r in summary:
    print(f"{r['sim']:6s} {r['z']:5s}  centrals={r['n_central']:6d}  "
          f"hosts_w_sat={r['n_host']:5d}  sats={r['n_sat']:6d}")